# Evaluación + videos del mejor modelo de CarRacing PPO

Esta notebook:

1. Reconstruye el mismo pipeline visual que usaste para entrenar.
2. Carga `best_model.zip`.
3. Evalúa el modelo en varios episodios.
4. Graba videos en la carpeta `videos/`.

> Antes de correr, ajustá `RUN_DIR` para que apunte a la carpeta de tu corrida.

In [1]:
%pip install "gymnasium[box2d]" stable-baselines3[extra] torch opencv-python pyyaml moviepy

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import os
import json
import numpy as np
import gymnasium as gym
import cv2

from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack, VecMonitor, VecTransposeImage, VecVideoRecorder


In [3]:
# ========= CONFIGURACIÓN =========
RUN_DIR = Path("experiments/carracing_ppo/run_20260425_052403_seed2")

# Si querés evaluar el mejor modelo:
MODEL_PATH = RUN_DIR / "best_model" / "best_model.zip"

# Si no existe best_model.zip, podés usar el final:
# MODEL_PATH = RUN_DIR / "final_model.zip"

ENV_ID = "CarRacing-v3"
SEED = 0

# Debe coincidir con el entrenamiento
CONTINUOUS = True
GRAYSCALE = True
N_STACK = 4
resize_observation = True
resize_shape = (84, 84)


# Evaluación
N_EVAL_EPISODES = 20
DETERMINISTIC = True

LAP_COMPLETE_PERCENT = 0.95

# Videos
VIDEO_DIR = RUN_DIR / "videos"
VIDEO_PREFIX = "best_model_eval"
VIDEO_EPISODE_LENGTH = 2000   # duración máxima por video
N_VIDEO_EPISODES = 3

assert RUN_DIR.exists(), f"No existe RUN_DIR: {RUN_DIR}"
assert MODEL_PATH.exists(), f"No existe MODEL_PATH: {MODEL_PATH}"
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_DIR:", RUN_DIR.resolve())
print("MODEL_PATH:", MODEL_PATH.resolve())
print("VIDEO_DIR:", VIDEO_DIR.resolve())


RUN_DIR: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2
MODEL_PATH: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\best_model\best_model.zip
VIDEO_DIR: C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\videos


In [4]:
# ========= WRAPPERS =========

class GrayscaleObservation(gym.ObservationWrapper):
    """Convierte RGB uint8 (H, W, 3) a grayscale uint8 (H, W, 1)."""

    def __init__(self, env: gym.Env):
        super().__init__(env)
        assert isinstance(env.observation_space, spaces.Box)
        obs_space = env.observation_space
        assert len(obs_space.shape) == 3 and obs_space.shape[-1] == 3, (
            f"Expected channel-last RGB image, got shape={obs_space.shape}"
        )
        h, w, _ = obs_space.shape
        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=(h, w, 1),
            dtype=np.uint8,
        )

    def observation(self, obs: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(obs, cv2.COLOR_RGB2GRAY)
        return gray[..., None].astype(np.uint8)
    

class ResizeObservation(gym.ObservationWrapper):
    def __init__(self, env: gym.Env, shape: tuple[int, int]):
        super().__init__(env)
        self.shape = shape

        old_space = env.observation_space
        assert isinstance(old_space, spaces.Box)

        channels = old_space.shape[2]
        new_shape = (shape[0], shape[1], channels)

        self.observation_space = spaces.Box(
            low=0,
            high=255,
            shape=new_shape,
            dtype=np.uint8,
        )

    def observation(self, obs: np.ndarray) -> np.ndarray:
        resized = cv2.resize(
            obs,
            (self.shape[1], self.shape[0]),
            interpolation=cv2.INTER_AREA,
        )
        if resized.ndim == 2:
            resized = resized[..., None]
        return resized.astype(np.uint8)


In [5]:
# ========= FACTORY DEL ENTORNO =========

def build_single_env(seed: int, monitor_file: Path | None = None, render_mode: str | None = None):
    env = gym.make(
        ENV_ID,
        continuous=CONTINUOUS,
        render_mode=render_mode,
        lap_complete_percent=LAP_COMPLETE_PERCENT,
    )

    env.action_space.seed(seed)
    env.reset(seed=seed)

    if GRAYSCALE:
        env = GrayscaleObservation(env)

    if resize_observation:
        env = ResizeObservation(env, resize_shape)

    if monitor_file is not None:
        env = Monitor(env, filename=str(monitor_file))
    else:
        env = Monitor(env)

    return env


def make_eval_vec_env(seed: int = 0, with_video: bool = False):
    render_mode = "rgb_array" if with_video else None

    def _init():
        return build_single_env(
            seed=seed,
            monitor_file=RUN_DIR / f"monitor_eval_notebook_seed{seed}.csv",
            render_mode=render_mode,
        )

    vec_env = DummyVecEnv([_init])
    vec_env = VecMonitor(vec_env, filename=str(RUN_DIR / "vec_monitor_eval_notebook.csv"))
    vec_env = VecFrameStack(vec_env, n_stack=N_STACK)
    vec_env = VecTransposeImage(vec_env)

    return vec_env


def unwrap_to_base_env(vec_env):
    current = vec_env
    while hasattr(current, "venv"):
        current = current.venv
    if hasattr(current, "envs") and len(current.envs) > 0:
        return current.envs[0].unwrapped
    raise RuntimeError("No se pudo acceder al entorno base.")


def episode_completion_status(vec_env):
    """Devuelve True si el episodio completó la vuelta.

    Se basa en la lógica oficial de CarRacing: la vuelta se considera completada
    cuando el porcentaje de tiles visitados supera `lap_complete_percent`.\n    """
    base_env = unwrap_to_base_env(vec_env)
    track = getattr(base_env, "track", None)
    tile_visited_count = getattr(base_env, "tile_visited_count", None)
    lap_complete_percent = getattr(base_env, "lap_complete_percent", LAP_COMPLETE_PERCENT)

    if track is None or tile_visited_count is None or len(track) == 0:
        return False

    visited_fraction = tile_visited_count / len(track)
    return visited_fraction >= lap_complete_percent


In [6]:
# ========= CARGA DEL MODELO =========

eval_env = make_eval_vec_env(seed=SEED, with_video=False)

print("Observation space:", eval_env.observation_space)
print("Action space:", eval_env.action_space)

model = PPO.load(MODEL_PATH, env=eval_env)

print("Modelo cargado correctamente.")


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Observation space: Box(0, 255, (4, 84, 84), uint8)
Action space: Box([-1.  0.  0.], 1.0, (3,), float32)
Modelo cargado correctamente.


In [7]:
# ========= EVALUACIÓN CUANTITATIVA =========

mean_reward, std_reward = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=N_EVAL_EPISODES,
    deterministic=DETERMINISTIC,
    render=False,
    return_episode_rewards=False,
)

print(f"Mean reward over {N_EVAL_EPISODES} eval episodes: {mean_reward:.3f}")
print(f"Std reward over  {N_EVAL_EPISODES} eval episodes: {std_reward:.3f}")
print("\nNota: el completion rate se calcula en la celda 'EVALUACIÓN EPISODIO POR EPISODIO'.")


Mean reward over 20 eval episodes: 849.800
Std reward over  20 eval episodes: 138.000

Nota: el completion rate se calcula en la celda 'EVALUACIÓN EPISODIO POR EPISODIO'.


In [8]:
# ========= EVALUACIÓN EPISODIO POR EPISODIO =========

import pandas as pd

episode_rewards = []
episode_lengths = []
episode_completed = []
records = []

for ep in range(N_EVAL_EPISODES):
    obs = eval_env.reset()
    total_reward = 0.0
    steps = 0

    while True:
        action, _states = model.predict(obs, deterministic=DETERMINISTIC)
        obs, rewards, dones, infos = eval_env.step(action)

        total_reward += float(rewards[0])
        steps += 1

        if dones[0]:
            break

    completed = episode_completion_status(eval_env)

    episode_rewards.append(total_reward)
    episode_lengths.append(steps)
    episode_completed.append(bool(completed))
    records.append({
        "episode": ep + 1,
        "reward": total_reward,
        "length": steps,
        "completed_lap": bool(completed),
    })

    print(
        f"Episode {ep+1}: reward={total_reward:.3f}, length={steps}, completed_lap={completed}"
    )

results_df = pd.DataFrame(records)
completion_rate = 100.0 * np.mean(episode_completed) if episode_completed else 0.0

print("\nResumen:")
print(f"Reward medio:      {np.mean(episode_rewards):.3f}")
print(f"Reward std:        {np.std(episode_rewards):.3f}")
print(f"Length medio:      {np.mean(episode_lengths):.3f}")
print(f"Completion rate:   {completion_rate:.1f}% ({sum(episode_completed)}/{len(episode_completed)})")

results_df


Episode 1: reward=925.300, length=747, completed_lap=False
Episode 2: reward=680.731, length=1000, completed_lap=False
Episode 3: reward=884.076, length=1000, completed_lap=False
Episode 4: reward=878.723, length=1000, completed_lap=False
Episode 5: reward=875.806, length=1000, completed_lap=False
Episode 6: reward=926.900, length=731, completed_lap=False
Episode 7: reward=917.200, length=828, completed_lap=False
Episode 8: reward=701.223, length=1000, completed_lap=False
Episode 9: reward=896.516, length=1000, completed_lap=False
Episode 10: reward=920.000, length=800, completed_lap=False
Episode 11: reward=868.750, length=1000, completed_lap=False
Episode 12: reward=923.100, length=769, completed_lap=False
Episode 13: reward=922.900, length=771, completed_lap=False
Episode 14: reward=929.200, length=708, completed_lap=False
Episode 15: reward=798.592, length=1000, completed_lap=False
Episode 16: reward=882.143, length=1000, completed_lap=False
Episode 17: reward=916.000, length=840, 

,episode,reward,length,completed_lap
0,1,925.300021,747,False
1,2,680.730875,1000,False
2,3,884.076458,1000,False
3,4,878.723400,1000,False
4,5,875.806466,1000,False
5,6,926.899980,731,False
6,7,917.200004,828,False
7,8,701.223221,1000,False
8,9,896.515661,1000,False
9,10,920.000004,800,False


In [9]:
# ========= GRABAR VIDEOS =========
# Esto crea archivos .mp4 dentro de RUN_DIR/videos

video_env = make_eval_vec_env(seed=SEED + 123, with_video=True)

video_env = VecVideoRecorder(
    video_env,
    video_folder=str(VIDEO_DIR),
    record_video_trigger=lambda step: step == 0,
    video_length=VIDEO_EPISODE_LENGTH,
    name_prefix=VIDEO_PREFIX,
)

video_model = PPO.load(MODEL_PATH, env=video_env)

for ep in range(N_VIDEO_EPISODES):
    obs = video_env.reset()
    total_reward = 0.0
    steps = 0

    while True:
        action, _ = video_model.predict(obs, deterministic=DETERMINISTIC)
        obs, rewards, dones, infos = video_env.step(action)

        total_reward += float(rewards[0])
        steps += 1

        if dones[0] or steps >= VIDEO_EPISODE_LENGTH:
            break

    print(f"Video episode {ep+1}: reward={total_reward:.3f}, length={steps}")

video_env.close()

print("\nVideos generados en:")
print(VIDEO_DIR.resolve())
print("\nArchivos:")
for p in sorted(VIDEO_DIR.glob("*.mp4")):
    print(" -", p.name)


c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\.venv\Lib\site-packages\stable_baselines3\common\vec_env\vec_monitor.py:43: UserWarning: The environment is already wrapped with a `Monitor` wrapperbut you are wrapping it with a `VecMonitor` wrapper, the `Monitor` statistics will beoverwritten by the `VecMonitor` ones.
  warnings.warn(


Video episode 1: reward=401.493, length=1000
Saving video to c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\videos\best_model_eval-step-0-to-step-2000.mp4
MoviePy - Building video c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\videos\best_model_eval-step-0-to-step-2000.mp4.
MoviePy - Writing video c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\videos\best_model_eval-step-0-to-step-2000.mp4



MoviePy - Done !
MoviePy - video ready c:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\videos\best_model_eval-step-0-to-step-2000.mp4
Video episode 2: reward=883.108, length=1000
Video episode 3: reward=893.631, length=1000

Videos generados en:
C:\Users\Jose\CEIA\Materias\Aprendizaje por Refuerzo I\Desafio\Codigo\ceia-ar1-desafio\gymnasium\carracing\ppo\12_resize_multiseed_seed2\experiments\carracing_ppo\run_20260425_052403_seed2\videos

Archivos:
 - best_model_eval-step-0-to-step-2000.mp4


In [10]:
# ========= OPCIONAL: VISUALIZAR QUÉ HAY EN LA CARPETA DE LA CORRIDA =========

print("Contenido principal de RUN_DIR:")
for p in sorted(RUN_DIR.iterdir()):
    print(" -", p.name)

print("\nVideos encontrados:")
for p in sorted(VIDEO_DIR.glob("*")):
    print(" -", p.name)


Contenido principal de RUN_DIR:
 - best_model
 - checkpoints
 - config.json
 - config.yaml
 - eval
 - final_model.zip
 - monitor_eval_10000.csv.monitor.csv
 - monitor_eval_notebook_seed0.csv.monitor.csv
 - monitor_eval_notebook_seed123.csv.monitor.csv
 - monitor_train_0.csv.monitor.csv
 - monitor_train_1.csv.monitor.csv
 - monitor_train_2.csv.monitor.csv
 - monitor_train_3.csv.monitor.csv
 - monitor_train_4.csv.monitor.csv
 - monitor_train_5.csv.monitor.csv
 - monitor_train_6.csv.monitor.csv
 - monitor_train_7.csv.monitor.csv
 - tb
 - vec_monitor_eval.csv.monitor.csv
 - vec_monitor_eval_notebook.csv.monitor.csv
 - vec_monitor_train.csv.monitor.csv
 - videos

Videos encontrados:
 - best_model_eval-step-0-to-step-2000.mp4


In [11]:
import numpy as np
import pandas as pd

evaluation_path = RUN_DIR / "eval" / "evaluations.npz"
data = np.load(evaluation_path)

timesteps = data["timesteps"]
results = data["results"]
ep_lengths = data["ep_lengths"]

mean_rewards = results.mean(axis=1)
std_rewards = results.std(axis=1)
min_rewards = results.min(axis=1)
max_rewards = results.max(axis=1)

mean_lengths = ep_lengths.mean(axis=1)
std_lengths = ep_lengths.std(axis=1)

df = pd.DataFrame({
    "timesteps": timesteps,
    "reward_mean": mean_rewards,
    "reward_std": std_rewards,
    "reward_min": min_rewards,
    "reward_max": max_rewards,
    "ep_length_mean": mean_lengths,
    "ep_length_std": std_lengths,
})

print(df)

    timesteps  reward_mean  reward_std  reward_min  reward_max  \
0       25000   -51.693798   49.789494  -95.538460   14.179893   
1       50000    -6.910375   22.355366  -27.927738   35.339638   
2       75000   325.397217  206.101105   26.317022  648.325928   
3      100000   472.641418  210.153259   77.359779  702.016174   
4      125000   475.862701  220.359360  267.645508  864.296387   
5      150000   508.499817  223.969742  164.578369  861.697205   
6      175000   549.463684  199.782639  301.814575  775.485535   
7      200000   608.020630  178.530304  375.347321  875.698242   
8      225000   595.237610  137.728470  384.532043  798.840088   
9      250000   617.694702  321.877563    6.215729  859.257996   
10     275000   452.252106  259.640594  207.931946  917.603577   
11     300000   413.454651  212.000900  135.772369  744.301514   
12     325000   535.826111  240.406433  249.382904  830.469360   
13     350000   584.600220  327.175140  -38.982838  876.115356   
14     375